In [ ]:
!git clone https://github.com/deepanrajm/GL_MML.git

In [2]:
import os
import librosa
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [3]:

def load_real_spectrograms(folder_path, target_shape=(64, 64)):
    data = []
    files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
    for f in files:
        y, sr = librosa.load(os.path.join(folder_path, f), duration=2.0)

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=target_shape[0])
        S_dB = librosa.power_to_db(S, ref=np.max)
        # Resize/Pad to ensure uniform shape for the CNN
        if S_dB.shape[1] < target_shape[1]:
            S_dB = np.pad(S_dB, ((0, 0), (0, target_shape[1] - S_dB.shape[1])))
        else:
            S_dB = S_dB[:, :target_shape[1]]
        data.append(S_dB)
    return np.array(data)

In [4]:

siren_path = "/content/GL_MML/Unit_2/Sound_Representation/siren/"
bark_path = "/content/GL_MML/Unit_2/Sound_Representation/dog_bark/"

sirens = load_real_spectrograms(siren_path)
barks = load_real_spectrograms(bark_path)

# Prepare for Training (Normalize to 0-1 range)
X = np.vstack((sirens, barks))
X = (X - X.min()) / (X.max() - X.min())
X = X.reshape(-1, 64, 64, 1) # Shape for CNN: (Samples, H, W, Channels)

In [5]:

# Encoder
enc_input = layers.Input(shape=(64, 64, 1))
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(enc_input)
x = layers.MaxPooling2D((2, 2), padding='same')(x)
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2), padding='same')(x)
x = layers.Flatten()(x)
latent_dim = layers.Dense(2, name="bottleneck")(x) # COMPRESSED TO 2D
encoder = models.Model(enc_input, latent_dim)

In [6]:
# Decoder
dec_input = layers.Input(shape=(2,))
x = layers.Dense(16 * 16 * 16, activation='relu')(dec_input)
x = layers.Reshape((16, 16, 16))(x)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = layers.UpSampling2D((2, 2))(x)
dec_output = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)
decoder = models.Model(dec_input, dec_output)

In [8]:
# Full Model
autoencoder = models.Model(enc_input, decoder(encoder(enc_input)))
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(X, X, epochs=100, batch_size=8, verbose=0)

In [ ]:
# 3. VISUALIZATION
embeddings = encoder.predict(X)
reconstructions = autoencoder.predict(X)


plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.scatter(embeddings[:len(sirens), 0], embeddings[:len(sirens), 1], label='Actual Sirens', c='blue', alpha=0.6)
plt.scatter(embeddings[len(sirens):, 0], embeddings[len(sirens):, 1], label='Actual Barks', c='orange', alpha=0.6)
plt.title("Latent Space: Where the AI stores sounds")
plt.xlabel("Latent Dim 1 (Coordinate X)")
plt.ylabel("Latent Dim 2 (Coordinate Y)")
plt.legend()
plt.show()


fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Sample 1: A Siren
axes[0, 0].imshow(X[0].reshape(64, 64), cmap='magma')
axes[0, 0].set_title("Original Siren Spectrogram")
axes[0, 0].axis('off')

axes[0, 1].imshow(reconstructions[0].reshape(64, 64), cmap='magma')
axes[0, 1].set_title("Reconstructed Siren")
axes[0, 1].axis('off')

# Sample 2: A Bark
bark_start_idx = len(sirens)
axes[1, 0].imshow(X[bark_start_idx].reshape(64, 64), cmap='magma')
axes[1, 0].set_title("Original Bark Spectrogram")
axes[1, 0].axis('off')

axes[1, 1].imshow(reconstructions[bark_start_idx].reshape(64, 64), cmap='magma')
axes[1, 1].set_title("Reconstructed Bark")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os
import librosa
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

# 1. DATA LOADING (Re-using your existing paths)
def load_real_spectrograms(folder_path, target_shape=(64, 64)):
    data = []
    files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]
    for f in files:
        y, sr = librosa.load(os.path.join(folder_path, f), duration=2.0)
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=target_shape[0])
        S_dB = librosa.power_to_db(S, ref=np.max)
        if S_dB.shape[1] < target_shape[1]:
            S_dB = np.pad(S_dB, ((0, 0), (0, target_shape[1] - S_dB.shape[1])))
        else:
            S_dB = S_dB[:, :target_shape[1]]
        data.append(S_dB)
    return np.array(data)

siren_path = "/content/GL_MML/Unit_2/Sound_Representation/siren/"
bark_path = "/content/GL_MML/Unit_2/Sound_Representation/dog_bark/"

sirens = load_real_spectrograms(siren_path)
barks = load_real_spectrograms(bark_path)

X = np.vstack((sirens, barks))
X = (X - X.min()) / (X.max() - X.min())
X = X.reshape(-1, 64, 64, 1)

# 2. VAE ARCHITECTURE
latent_dim = 2

class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = K.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# Encoder
enc_input = layers.Input(shape=(64, 64, 1))
x = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(enc_input)
x = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation="relu")(x)

z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])

encoder = models.Model(enc_input, [z_mean, z_log_var, z], name="encoder")

# Decoder
latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(16 * 16 * 64, activation="relu")(latent_inputs)
x = layers.Reshape((16, 16, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
dec_output = layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(x)

decoder = models.Model(latent_inputs, dec_output, name="decoder")

class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(tf.reduce_sum(
                tf.keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2)))
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        return {"loss": total_loss, "reconstruction_loss": reconstruction_loss, "kl_loss": kl_loss}

# Train the model
vae = VAE(encoder, decoder)
vae.compile(optimizer='adam')
vae.fit(X, epochs=100, batch_size=8, verbose=0)

# 4. VISUALIZATION
# 1. Get Latent Representations and Reconstructions
z_mean, z_log_var, _ = encoder.predict(X)
reconstructions = decoder.predict(z_mean)

# 2. Setup the Visualization Plot
plt.figure(figsize=(18, 20))

# --- PART A: LATENT SPACE (The Organized Map) ---
plt.subplot(3, 2, 1) # Changed from 2,2,1 to 3,2,1
plt.scatter(z_mean[:len(sirens), 0], z_mean[:len(sirens), 1], label='Sirens', alpha=0.6, c='blue')
plt.scatter(z_mean[len(sirens):, 0], z_mean[len(sirens):, 1], label='Barks', alpha=0.6, c='orange')
plt.title("VAE Latent Space: Organised Sound Distributions")
plt.xlabel("Latent Dim 1 (Mean)")
plt.ylabel("Latent Dim 2 (Mean)")
plt.legend()

# --- PART B: ORIGINAL VS RECONSTRUCTED (Siren) ---
plt.subplot(3, 2, 3) # Changed from 2,2,3 to 3,2,3
plt.imshow(X[0].reshape(64, 64), cmap='magma')
plt.title("Original Siren Spectrogram")
plt.axis('off')

plt.subplot(3, 2, 4)
plt.imshow(reconstructions[0].reshape(64, 64), cmap='magma')
plt.title("VAE Reconstructed Siren")
plt.axis('off')

# --- PART C: ORIGINAL VS RECONSTRUCTED (Bark) ---
bark_idx = len(sirens)
plt.subplot(3, 2, 5) # Changed from 2,2,5 to 3,2,5
plt.imshow(X[bark_idx].reshape(64, 64), cmap='magma')
plt.title("Original Bark Spectrogram")
plt.axis('off')


plt.subplot(3, 2, 6) # Changed from 2,2,6 to 3,2,6
plt.imshow(reconstructions[bark_idx].reshape(64, 64), cmap='magma')
plt.title("VAE Reconstructed Bark")
plt.axis('off')

plt.tight_layout()
plt.show()